# 02 — Modèles de Baseline
**TF-IDF + Logistic Regression + Naive Bayes**

Pipeline :
1. Preprocessing NLP classique (lowercase, ponctuation, stopwords, lemmatisation)
2. Vectorisation TF-IDF
3. Entraînement Logistic Regression & Multinomial Naive Bayes
4. Classification report & matrice de confusion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.pipeline import Pipeline

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 110

SEED = 42

## 1. Chargement

In [ ]:
df = pd.read_csv(
    '../data/all-data.csv',
    encoding='latin-1',
    header=None,
    names=['sentiment', 'headline']
)

print(f'Shape : {df.shape}')
print(df['sentiment'].value_counts())
df.head()

## 2. Preprocessing NLP

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess(text: str) -> str:
    """Lowercase → suppression ponctuation/chiffres → stopwords → lemmatisation."""
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)      # supprime ponctuation & chiffres
    text = re.sub(r'\s+', ' ', text).strip()    # espaces multiples
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['clean'] = df['headline'].apply(preprocess)

print('Exemple avant :', df['headline'].iloc[0])
print('Exemple après :', df['clean'].iloc[0])

## 3. Split train / test

In [ ]:
label_map = {'positive': 0, 'neutral': 1, 'negative': 2}
df['label'] = df['sentiment'].map(label_map)

X = df['clean']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train : {len(X_train)} | Test : {len(X_test)}')
print('Distribution test :', y_test.value_counts().to_dict())

## 4. TF-IDF Vectorisation

In [ ]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),   # unigrams + bigrams
    max_features=20_000,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Matrice TF-IDF train : {X_train_tfidf.shape}')
print(f'Vocabulaire : {len(tfidf.vocabulary_)} termes')

## 5. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
lr.fit(X_train_tfidf, y_train)

y_pred_lr = lr.predict(X_test_tfidf)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f'Accuracy Logistic Regression : {acc_lr:.4f}')
print('\n--- Classification Report ---')
target_names = ['positive', 'neutral', 'negative']
print(classification_report(y_test, y_pred_lr, target_names=target_names))

## 6. Naive Bayes

In [ ]:
nb = MultinomialNB(alpha=0.1)
nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)
acc_nb = accuracy_score(y_test, y_pred_nb)

print(f'Accuracy Naive Bayes : {acc_nb:.4f}')
print('\n--- Classification Report ---')
print(classification_report(y_test, y_pred_nb, target_names=target_names))

## 7. Matrices de confusion

In [ ]:
def plot_confusion(y_true, y_pred, model_name, ax, labels):
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    annot = np.array([[f'{v}\n({p:.0%})' for v, p in zip(row_v, row_p)]
                      for row_v, row_p in zip(cm, cm_pct)])
    sns.heatmap(
        cm, annot=annot, fmt='', cmap='Blues',
        xticklabels=labels, yticklabels=labels,
        linewidths=0.5, ax=ax
    )
    ax.set_title(f'{model_name}\nAcc = {accuracy_score(y_true, y_pred):.3f}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_confusion(y_test, y_pred_lr, 'Logistic Regression', axes[0], target_names)
plot_confusion(y_test, y_pred_nb, 'Naive Bayes',         axes[1], target_names)
plt.suptitle('Matrices de confusion — Baseline', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 8. Cross-validation & comparaison

In [ ]:
# Pipeline complet pour la CV
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=20_000, sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=1.0, random_state=SEED))
])

pipe_nb = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=20_000, sublinear_tf=True)),
    ('clf',   MultinomialNB(alpha=0.1))
])

cv_lr = cross_val_score(pipe_lr, X, y, cv=5, scoring='accuracy')
cv_nb = cross_val_score(pipe_nb, X, y, cv=5, scoring='accuracy')

results = pd.DataFrame({
    'Modèle'       : ['Logistic Regression', 'Naive Bayes'],
    'Acc Test'     : [acc_lr, acc_nb],
    'CV Mean (5f)' : [cv_lr.mean(), cv_nb.mean()],
    'CV Std'       : [cv_lr.std(),  cv_nb.std()]
})

print('=== Résultats Baseline ===')
print(results.round(4).to_string(index=False))

In [ ]:
# Sauvegarde des scores pour le notebook 04
import json, os
os.makedirs('../results', exist_ok=True)
scores = {
    'LogisticRegression': round(float(acc_lr), 4),
    'NaiveBayes':         round(float(acc_nb), 4)
}
with open('../results/baseline_scores.json', 'w') as f:
    json.dump(scores, f, indent=2)
print('Scores sauvegardés dans results/baseline_scores.json')